In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import warnings
warnings.simplefilter(action='ignore')

## Lecture 23 Review (Confidence Intervals)

In [ ]:
# Import San Francisco employment data
# Keep rows where salary is above minimum wage

sf = Table.read_table('san_francisco_2019.csv')
min_salary = 15 * 20 * 50
sf = sf.where('Salary', are.above(min_salary))

In [ ]:
# Population distribution of total compensation

sf_bins = np.arange(0, 726000, 25000)
sf.hist('Total Compensation', bins=sf_bins)

In [ ]:
# Parameter: Median total compensation in the population

def median_comp(t):
    return percentile(50, t.column('Total Compensation'))

# Population median total compensation
median_comp(sf)

In [ ]:
# Collect one random sample (n = 400) from the population

our_sample = sf.sample(400, with_replacement=False)

In [ ]:
# Generate the medians of 3000 bootstrap samples
num_repetitions = 3000
bstrap_medians = make_array()
for i in np.arange(num_repetitions):
# Resample the same number of rows, with replacement
    bstrap_medians = np.append(bstrap_medians, median_comp(our_sample.sample()))

# Find the middle 95% of medians; that's a confidence interval
left = percentile(2.5, bstrap_medians)
right = percentile(97.5, bstrap_medians)
make_array(left, right)

In [ ]:
resampled_medians = Table().with_column('Bootstrapped Sample Median', bstrap_medians)
median_bins=np.arange(120000, 160000, 2000)
resampled_medians.hist(bins = median_bins)

# Plotting parameters; you can ignore this code
parameter_green = '#32CD32'
plots.ylim(-0.000005, 0.00014)
plots.plot(make_array(left, right), make_array(0, 0), color='red', lw=3, zorder=1)
plots.scatter(median_comp(sf), 0, color=parameter_green, s=40, zorder=2);

### Bootstrap Estimates of the Parameter (Pretend it is Unknown) 

In [ ]:
def confidence_interval_95(sample_size):
    # Collect one random sample from the population
    our_sample = sf.sample(sample_size, with_replacement=False)
    median_comp(our_sample)

    # Generate the medians of 3000 bootstrap samples
    num_repetitions = 3000
    bstrap_medians = make_array()
    for i in np.arange(num_repetitions):
        # Resample the same number of rows, with replacement
        bstrap_medians = np.append(bstrap_medians, 
                                   median_comp(our_sample.sample()))

    # Find the middle 95% of medians; that's a confidence interval
    left = percentile(2.5, bstrap_medians)
    right = percentile(97.5, bstrap_medians)
    return make_array(left, right)

In [ ]:
# 95% confidence interval using sample size 400
# Note different confidence interval if run again

...

In [ ]:
resampled_medians = Table().with_column('Bootstrapped Sample Median', bstrap_medians)
median_bins=np.arange(120000, 160000, 2000)
resampled_medians.hist(bins = median_bins)

# Plotting parameters; you can ignore this code
parameter_green = '#32CD32'
plots.ylim(-0.000005, 0.00014)
plots.plot(make_array(left, right), make_array(0, 0), color='red', lw=3, zorder=1)
plots.scatter(median_comp(sf), 0, color=parameter_green, s=40, zorder=2);

## Lecture 24 (Interpreting Confidence)

In [ ]:
# This takes a while to run (2-3 minutes)

# 95% confidence interval is not 100% reliable <-- 100% reliable means every CI contains the parameter.
# Repeat several times to see how often it contains the true population parameter
# Sample size 400, 20 times

intervals = Table(['k', 'lower', 'upper', 'sample_size'])

for k in np.arange(20):
    interval = confidence_interval_95(400)
    intervals = intervals.with_row([k, interval.item(0), interval.item(1), 400])
    
intervals.show(5)

In [ ]:
# A good confidence interval contains the
# true population parameter and is narrow

# How many of the confidence intervals
# contain the population parameter?

intervals.scatter('lower', 'k', group='sample_size')
truth = median_comp(sf)
plots.plot((truth, truth), (0, intervals.num_rows), color='black', lw=3)
plots.xticks([120000, 150000])
for row in intervals.rows:
    if row.item('lower') > truth or row.item('upper') < truth:
        color = 'red'
    else:
        color = 'blue'
    plots.plot((row.item('lower'), row.item('upper')),
               (row.item('k'), row.item('k')),
               color, lw=2)


In [ ]:
# This takes a while to run (4-5 minutes)

# 95% confidence interval is not 100% reliable 
# Repeat several times to see how often it
# contains the true population parameter
# Sample size 100, 20 times
# Sample size 400, 20 times

intervals = Table(['k', 'lower', 'upper', 'sample_size'])

for n in [100, 400]:
    for k in np.arange(intervals.num_rows, intervals.num_rows + 20):
        interval = confidence_interval_95(n)
        intervals = intervals.with_row([
            k, interval.item(0), interval.item(1), n
        ])
    
intervals.show(5)

In [ ]:
# A good confidence interval contains the
# true population parameter and is narrow

# How many of the confidence intervals
# contain the population parameter?

intervals.scatter('lower', 'k', group='sample_size')
truth = median_comp(sf)
plots.plot((truth, truth), (0, intervals.num_rows), color='green', lw=3)
plots.xticks([120000, 150000])
for row in intervals.rows:
    if row.item('lower') > truth or row.item('upper') < truth:
        color = 'red'
    else:
        color = 'blue'
    plots.plot((row.item('lower'), row.item('upper')),
               (row.item('k'), row.item('k')),
               color, lw=2)


## Confidence Interval for Unknown Population Mean

In [ ]:
# Random sample of mother-newborn pairs
births = Table.read_table('baby.csv')
births

In [ ]:
births.hist('Maternal Age')

In [ ]:
# Average age of mothers in the sample

np.average(births.column('Maternal Age'))

### Question
What is the average age of the mothers in the population?

In [ ]:
def one_bootstrap_mean():
    resample = births...
    return ...

In [ ]:
# Generate means from 3000 bootstrap samples
num_repetitions = 3000
bstrap_means = make_array()
for i in np.arange(...):
    bstrap_means = np.append(bstrap_means, ...)

### Bootstrap Percentile Method for Confidence Interval

The interval of estimates is the "middle 95%" of the bootstrapped estimates.

This is called a *95% confidence interval* for the mean age in the population.

In [ ]:
# Get the endpoints of the 95% confidence interval
left = ...
right = ...

make_array(left, right)

In [ ]:
resampled_means = Table().with_columns(
    'Bootstrap Sample Mean', bstrap_means
)
resampled_means.hist(bins=15)
plots.plot([left, right], [0, 0], color='red', lw=8);